# Exploration 4 — Step 1: Precompute I and R

Compute and save the two matrices needed by the QUBO:
- `importance.csv` — I[i,j] = Var_k[ RSS(k, i, j) ]  for all off-diagonal (i,j)
- `redundancy.csv` — R[p, q] = |Pearson corr| between link p and link q  (90×90)

Also saves `pairs.csv` — the ordered list of (i,j) pairs mapping variable index p → (AP=i, MP=j).

## 0. Imports & Paths

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA  = Path('../../rti/basement')
OUT   = Path('data')
OUT.mkdir(exist_ok=True)

N       = 10
NUM_CHS = 8

print('Data folder exists:', DATA.exists())

Data folder exists: True


## 1. Load Raw Data

In [2]:
sensor_coords = np.loadtxt(DATA / 'sensor_coords_basement_m.txt')   # (10, 2)
pivot_coords  = np.loadtxt(DATA / 'pivot_coords_basement_m.txt')    # (21, 2)
path1         = np.loadtxt(DATA / 'path_basement_1_f.txt', dtype=int)
path2         = np.loadtxt(DATA / 'path_basement_2_f.txt', dtype=int)

raw1 = np.loadtxt(DATA / 'basement_listenx_out_1.txt')
raw2 = np.loadtxt(DATA / 'basement_listenx_out_2.txt')
rss1 = raw1[:, :-1]   # strip timestamp
rss2 = raw2[:, :-1]

print(f'RSS run 1: {rss1.shape},  run 2: {rss2.shape}')

RSS run 1: (642, 720),  run 2: (635, 720)


## 2. Build RSS Tensor  (T × N × N, channel-averaged)

In [3]:
def build_rss_tensor(rss_mat, N=10, NUM_CHS=8):
    T = rss_mat.shape[0]
    rss_3d     = rss_mat.reshape(T, NUM_CHS, N * (N - 1))
    rss_ch_avg = rss_3d.mean(axis=1)                        # (T, 90)
    tensor = np.full((T, N, N), np.nan)
    for link_idx in range(N * (N - 1)):
        tx = link_idx // (N - 1)
        rx = link_idx  % (N - 1)
        if rx >= tx:
            rx += 1
        tensor[:, tx, rx] = rss_ch_avg[:, link_idx]
    return tensor

tensor1 = build_rss_tensor(rss1)
tensor2 = build_rss_tensor(rss2)
print(f'Tensor run 1: {tensor1.shape},  run 2: {tensor2.shape}')

Tensor run 1: (642, 10, 10),  run 2: (635, 10, 10)


## 3. Build Fingerprint Database  (K × N × N, per unique pivot)

In [4]:
def build_fingerprint_db(tensor, path, pivot_coords, N=10):
    T, S = tensor.shape[0], len(path)
    step_edges = np.round(np.linspace(0, T, S + 1)).astype(int)
    step_rss = np.full((S, N, N), np.nan)
    for s in range(S):
        t0, t1 = step_edges[s], step_edges[s + 1]
        if t1 > t0:
            step_rss[s] = np.nanmean(tensor[t0:t1], axis=0)
    unique_pivots = np.unique(path)
    K = len(unique_pivots)
    fp_rss    = np.full((K, N, N), np.nan)
    fp_coords = np.zeros((K, 2))
    for k, piv in enumerate(unique_pivots):
        mask = (path == piv)
        fp_rss[k]    = np.nanmean(step_rss[mask], axis=0)
        fp_coords[k] = pivot_coords[piv]
    return fp_rss, fp_coords, unique_pivots

fp_rss1, fp_coords1, fp_pivots1 = build_fingerprint_db(tensor1, path1, pivot_coords)
fp_rss2, fp_coords2, fp_pivots2 = build_fingerprint_db(tensor2, path2, pivot_coords)

print(f'Fingerprint DB — train: {fp_rss1.shape},  test: {fp_rss2.shape}')
print(f'K = {fp_rss1.shape[0]} unique fingerprint locations')

Fingerprint DB — train: (21, 10, 10),  test: (21, 10, 10)
K = 21 unique fingerprint locations


C:\Users\WIN 11\AppData\Local\Temp\ipykernel_53524\3647577446.py:8: RuntimeWarning: Mean of empty slice
  step_rss[s] = np.nanmean(tensor[t0:t1], axis=0)
C:\Users\WIN 11\AppData\Local\Temp\ipykernel_53524\3647577446.py:15: RuntimeWarning: Mean of empty slice
  fp_rss[k]    = np.nanmean(step_rss[mask], axis=0)


## 4. Compute Importance  I[i,j] = Var_k[ RSS(k, i, j) ]

Computed on the **training** fingerprint DB (run 1).

In [5]:
# All valid (off-diagonal) pairs in a fixed order
pairs = [(i, j) for i in range(N) for j in range(N) if i != j]   # 90 pairs
L = len(pairs)
print(f'Number of off-diagonal pairs (variables): {L}')

# I[p] = variance of RSS across K fingerprint locations for pair p
I = np.array([np.nanvar(fp_rss1[:, i, j]) for i, j in pairs])    # (90,)

print(f'Importance range: [{I.min():.4f}, {I.max():.4f}] dBm²')
print(f'Mean importance : {I.mean():.4f} dBm²')
print(f'Best pair       : AP={pairs[I.argmax()][0]} → MP={pairs[I.argmax()][1]}  (I={I.max():.4f})')

Number of off-diagonal pairs (variables): 90
Importance range: [0.9324, 20.9919] dBm²
Mean importance : 8.4527 dBm²
Best pair       : AP=1 → MP=6  (I=20.9919)


## 5. Compute Redundancy  R[p, q] = |Pearson corr( RSS_p, RSS_q )|

Each link p is a vector of K RSS values across fingerprint locations.

In [6]:
# Build link matrix: (K, L) — each column = RSS vector of one link
K = fp_rss1.shape[0]
link_matrix = np.array([fp_rss1[:, i, j] for i, j in pairs]).T   # (K, L)

# Absolute Pearson correlation matrix
col_labels = [f'({i},{j})' for i, j in pairs]
R = pd.DataFrame(link_matrix, columns=col_labels).corr().abs().values   # (L, L)

# Off-diagonal stats
mask_off = ~np.eye(L, dtype=bool)
R_off = R[mask_off]
print(f'Redundancy matrix shape : {R.shape}')
print(f'R range (off-diagonal)  : [{R_off.min():.4f}, {R_off.max():.4f}]')
print(f'Mean off-diagonal |r|   : {R_off.mean():.4f}')
print(f'Fraction |r| > 0.8      : {(R_off > 0.8).mean():.2%}')

Redundancy matrix shape : (90, 90)
R range (off-diagonal)  : [0.0000, 0.9702]
Mean off-diagonal |r|   : 0.4325
Fraction |r| > 0.8      : 3.70%


## 6. Save to CSV

In [7]:
# --- pairs.csv: variable index → (ap, mp) mapping ---
df_pairs = pd.DataFrame({'var_index': range(L),
                          'ap': [p[0] for p in pairs],
                          'mp': [p[1] for p in pairs]})
df_pairs.to_csv(OUT / 'pairs.csv', index=False)

# --- importance.csv: var_index, ap, mp, importance ---
df_importance = df_pairs.copy()
df_importance['importance'] = I
df_importance.to_csv(OUT / 'importance.csv', index=False)

# --- redundancy.csv: 90×90 matrix, rows/cols labelled by var_index ---
df_redundancy = pd.DataFrame(R,
                              index=range(L),
                              columns=range(L))
df_redundancy.index.name   = 'var_index'
df_redundancy.columns.name = 'var_index'
df_redundancy.to_csv(OUT / 'redundancy.csv')

# --- fingerprint coords (needed for KNN evaluation) ---
pd.DataFrame(fp_coords1, columns=['x', 'y']).to_csv(OUT / 'fp_coords_train.csv', index=False)
pd.DataFrame(fp_coords2, columns=['x', 'y']).to_csv(OUT / 'fp_coords_test.csv',  index=False)

# --- fingerprint RSS tensors (K × L) —- flat link order matches pairs.csv ---
fp_rss1_flat = np.array([fp_rss1[:, i, j] for i, j in pairs]).T   # (K, L)
fp_rss2_flat = np.array([fp_rss2[:, i, j] for i, j in pairs]).T
pd.DataFrame(fp_rss1_flat, columns=range(L)).to_csv(OUT / 'fp_rss_train.csv', index=False)
pd.DataFrame(fp_rss2_flat, columns=range(L)).to_csv(OUT / 'fp_rss_test.csv',  index=False)

print('Saved to', OUT.resolve())
for f in sorted(OUT.iterdir()):
    print(f'  {f.name}')

Saved to C:\Users\WIN 11\Desktop\AUC\4. Senior Year\Spring 2026\Senior Project II\Thesis 2\Exploration 4\Implementation\data
  fp_coords_test.csv
  fp_coords_train.csv
  fp_rss_test.csv
  fp_rss_train.csv
  importance.csv
  pairs.csv
  redundancy.csv


## 7. Quick Sanity Check

In [8]:
# Reload and verify
df_I  = pd.read_csv(OUT / 'importance.csv')
df_R  = pd.read_csv(OUT / 'redundancy.csv', index_col=0)
df_p  = pd.read_csv(OUT / 'pairs.csv')

assert len(df_I) == L,               f'importance rows: {len(df_I)} != {L}'
assert df_R.shape == (L, L),         f'redundancy shape: {df_R.shape} != ({L},{L})'
assert df_I['importance'].min() > 0, 'all importances should be > 0'
assert (df_R.values.diagonal() == 1).all(), 'diagonal of R should be 1'

print('All checks passed.')
print()
print('importance.csv — first 5 rows:')
print(df_I.head())
print()
print('redundancy.csv — top-left 5×5:')
print(df_R.iloc[:5, :5].round(4))

All checks passed.

importance.csv — first 5 rows:
   var_index  ap  mp  importance
0          0   0   1    6.935430
1          1   0   2    4.044727
2          2   0   3    8.848610
3          3   0   4    8.353595
4          4   0   5    2.127035

redundancy.csv — top-left 5×5:
                0       1       2       3       4
var_index                                        
0          1.0000  0.6499  0.3684  0.3437  0.5014
1          0.6499  1.0000  0.4811  0.6076  0.5255
2          0.3684  0.4811  1.0000  0.4213  0.4098
3          0.3437  0.6076  0.4213  1.0000  0.6007
4          0.5014  0.5255  0.4098  0.6007  1.0000
